In [1]:
# Imports
import numpy as np
import pandas as pd
import requests
import joblib
from datetime import datetime, timezone

In [2]:
# The 13 features the deployed models use, all available from the weather API
features_weather_only = ['GHI', 'temp', 'pressure', 'humidity', 'wind_speed',
                          'rain_1h', 'snow_1h', 'clouds_all', 'dayLength',
                          'hour_sin', 'hour_cos', 'month_sin', 'month_cos']

# Horizons the saved models cover
allowed_horizons = ['15min', '30min', '45min', '1hour', '75min', '90min', '105min', '2hour']

In [3]:
# Cities supported by this pipeline, mapped to (lat, lon)
allowed_cities = {'Warsaw': (52.23, 21.01), 'Berlin': (52.52, 13.40), 'Amsterdam': (52.37, 4.90),
                   'London': (51.51, -0.13), 'Hamburg': (53.55, 9.99)}

In [4]:
# Fetch hourly weather for a supported city at a specific UTC hour
def get_weather_at(city_name, current_datetime):
    city_name = city_name.strip().title()
    if city_name not in allowed_cities:
        print('Error: city not found in allowed_cities:', city_name)
        return None

    lat, lon = allowed_cities[city_name]
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': lat,
        'longitude': lon,
        'hourly': 'temperature_2m,relative_humidity_2m,pressure_msl,cloud_cover,wind_speed_10m,precipitation,snowfall,shortwave_radiation,is_day',
        'wind_speed_unit': 'ms',
        'timezone': 'UTC',
        'forecast_days': 7
    }
    response = requests.get(url, params=params)
    data = response.json()
    hourly = data['hourly']

    wanted = current_datetime.strftime('%Y-%m-%dT%H:00')
    if wanted not in hourly['time']:
        print('Error: requested time is outside the API window:', wanted)
        return None
    index = hourly['time'].index(wanted)

    ghi_api = hourly['shortwave_radiation'][index]
    if ghi_api is None:
        print('Error: GHI (shortwave_radiation) is missing for', city_name)
        return None

    weather = {
        'temp': hourly['temperature_2m'][index],
        'humidity': hourly['relative_humidity_2m'][index],
        'pressure': hourly['pressure_msl'][index],
        'clouds_all': hourly['cloud_cover'][index],
        'wind_speed': hourly['wind_speed_10m'][index],
        'rain_1h': hourly['precipitation'][index],
        'snow_1h': hourly['snowfall'][index] * 10,   # API returns cm, training data is mm
        'GHI': ghi_api / 4,                          # API returns W/m2, training data is Wh/m2 per 15-min step
        'is_day': hourly['is_day'][index]
    }
    return weather

In [5]:
# Approximate day length and cyclical hour/month features for a given datetime
def compute_time_features(current_datetime):
    latitude = 52
    day_of_year = current_datetime.timetuple().tm_yday
    declination = 23.45 * np.sin(np.deg2rad(360 * (284 + day_of_year) / 365))
    lat_rad = np.deg2rad(latitude)
    decl_rad = np.deg2rad(declination)
    hour_angle = np.arccos(-np.tan(lat_rad) * np.tan(decl_rad))
    day_length = (2 * hour_angle) * 24 / (2 * np.pi) * 60   # minutes, to match the training data

    hour = current_datetime.hour
    month = current_datetime.month
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    month_sin = np.sin(2 * np.pi * month / 12)
    month_cos = np.cos(2 * np.pi * month / 12)

    return day_length, hour_sin, hour_cos, month_sin, month_cos

In [6]:
# Predict Power[kW] for a city, horizon and UTC time
def get_forecast(city_name, horizon_choice, current_datetime):
    if horizon_choice not in allowed_horizons:
        print('Error: horizon not supported:', horizon_choice)
        return None

    weather = get_weather_at(city_name, current_datetime)
    if weather is None:
        return None

    if weather['is_day'] == 0 or weather['GHI'] <= 0.1:
        print("It's nighttime — no solar output expected")
        return 0.0

    day_length, hour_sin, hour_cos, month_sin, month_cos = compute_time_features(current_datetime)

    row = pd.DataFrame([{
        'GHI': weather['GHI'],
        'temp': weather['temp'],
        'pressure': weather['pressure'],
        'humidity': weather['humidity'],
        'wind_speed': weather['wind_speed'],
        'rain_1h': weather['rain_1h'],
        'snow_1h': weather['snow_1h'],
        'clouds_all': weather['clouds_all'],
        'dayLength': day_length,
        'hour_sin': hour_sin,
        'hour_cos': hour_cos,
        'month_sin': month_sin,
        'month_cos': month_cos
    }])
    row = row[features_weather_only]

    model = joblib.load(f'../Models/models/forecast_weather_only_{horizon_choice}.joblib')
    prediction = model.predict(row)
    return prediction[0]

In [7]:
# Sanity check: units must match the training data
print('Longest day  (expect ~1020):', round(compute_time_features(datetime(2026, 6, 21, 12, 0))[0]))
print('Shortest day (expect ~450):', round(compute_time_features(datetime(2026, 12, 21, 12, 0))[0]))

w = get_weather_at('Warsaw', datetime(2026, 9, 19, 10, 0))
print('GHI (expect under 229):', w['GHI'])
print('Wind in m/s (expect under 15):', w['wind_speed'])

Longest day  (expect ~1020): 990
Shortest day (expect ~450): 450
GHI (expect under 229): 141.0
Wind in m/s (expect under 15): 3.2


In [8]:
# Solar noon in Warsaw is around 10:00 UTC
result = get_forecast('Warsaw', '30min', datetime(2026, 9, 19, 10, 0))
print(f"Warsaw midday: {result:.2f} kW")

# Night
result = get_forecast('Warsaw', '30min', datetime(2026, 9, 19, 22, 0))
print(f"Warsaw night: {result:.2f} kW")

# Longer horizon, another city
result = get_forecast('Berlin', '2hour', datetime(2026, 9, 19, 11, 0))
print(f"Berlin in 2 hours: {result:.2f} kW")

Warsaw midday: 14.43 kW
It's nighttime — no solar output expected
Warsaw night: 0.00 kW
Berlin in 2 hours: 3.01 kW
